In [ ]:
import pandas as pd
import numpy as np

Units = pd.read_csv("./data/Units.csv")
Units = Units[Units["Status"]=='operating']
cond_oil = Units["Technology"].eq("oil")
Units.loc[cond_oil, "type"] = "oil"


Units = Units[Units["type"]!="cascade"]

BASE_MAP = {
    "offwind": "offshore_wind",
    "onwind":  "onshore_wind",
    "gas":     "ccgt",
    "biomass": "biomass",
    "fossil":   "ccgt",
    "oil":     "oil",
    "nuclear": "nuclear",
}

SOLAR_BUCKETS = [
    (0.0,     0.004,  "PV(<4kW)"),     # <50 kW
    (0.004,    0.01,   "PV(4-10kW)"),     # 50 kW–1 MW
    (0.01,     0.05,   "PV(10-50kW)"),    # 1–5 MW
    (0.05,     np.inf,"PV(50kW+)"),   # ≥5 MW
]

ONWIND_BUCKETS = [
    (0.0,     1.0,  "onshore_wind(<1MW))"),     # <1 MW
    (1.0,     3.0,   "onshore_wind(1-3MW)"),     # 1 MW – 3 MW
    (3.0,    np.inf,   "onshore_wind(>3MW)"),    # >3 MW
]

Units

def pick_solar_tech(cap_mw: float) -> str:
    for lo, hi, name in SOLAR_BUCKETS:
        if lo <= cap_mw < hi:
            return name
    return SOLAR_BUCKETS[-1][2]

def pick_wind_tech(cap_mw: float) -> str:
    for lo, hi, name in ONWIND_BUCKETS:
        if lo <= cap_mw < hi:
            return name
    return ONWIND_BUCKETS[-1][2]

def normalize_type(t: str, cap_mw: float) -> str:
    t_norm = str(t).strip().lower()
    if "solar" in t_norm or t_norm == "pv":
        return pick_solar_tech(cap_mw)
    if "onwind" in t_norm or t_norm == "onshore_wind":
        return pick_wind_tech(cap_mw)
    return BASE_MAP.get(t_norm, t_norm)

Units["tech"] = Units.apply(lambda r: normalize_type(r["type"], float(r["capacity"])), axis=1)
Units = Units[["name", "tech", "capacity"]]
Units.rename(columns={"name": "id", "capacity": "capacity_mw"}, inplace=True)
Units.to_csv("./data/units_test.csv", index=False)

In [4]:
Units

,id,tech,capacity_mw
0,DRAXD-2,biomass,2580.0
10,biomass_27,biomass,43.0
11,biomass_120,biomass,34.0
12,biomass_181,biomass,30.0
13,biomass_199,biomass,40.0
...,...,...,...
4413,gas_9472,ccgt,21.0
4414,gas_9473,ccgt,450.0
4415,gas_9474,ccgt,460.0
4416,gas_9475,ccgt,450.0
